---
title: Spec-Driven Agentic Coding
abstract: |
    This guide teaches a structured workflow for building software with AI agents: start by defining a spec through conversation, set up an isolated workspace with testing and logging, then let the agent implement, test, debug, and iterate autonomously. It covers end-user-first planning, environment setup, and the self-correcting agent loop.
---

Spec-driven agentic coding is a systematic approach to building software with AI coding agents like Hermes. Instead of asking the agent to "just write it" and hoping for the best, you follow a disciplined process:

1. **Define what to build** — chat with the agent to refine requirements from the end-user perspective
2. **Lock in the spec** — turn the conversation into a concrete implementation plan
3. **Set up the workspace** — create a repository, install dependencies, configure tests and logging
4. **Agent implements and self-corrects** — the agent writes code, runs tests, reads logs, finds errors, and fixes them until everything passes

This approach turns the agent from a code generator into an **autonomous developer** that can build, test, and debug on its own.

## Why Spec-Driven?

Most people use coding agents reactively — they describe a problem, get code back, and manually fix issues. Spec-driven coding is **proactive**: you invest time upfront to define clear requirements and acceptance criteria, then the agent works toward that target with measurable feedback.

Benefits:

- **Predictable results** — the agent has a concrete target, not a vague prompt
- **Self-healing** — automated tests catch regressions; the agent reads logs and fixes its own errors
- **Traceable** — every change ties back to a spec item, so you can audit why code exists
- **Scalable** — works for a single utility script or a multi-file project with dependencies

Without a spec and automated feedback, the agent is just guessing what you want. With them, it's building to a blueprint.

## Phase 1: Define the Spec

### Start From the End-User's Perspective

The most common mistake is describing what you want in technical terms before understanding what the user actually needs. Instead, start by describing the problem from the end-user's point of view:

**Bad spec:** "Create a Flask API with `/bmi` endpoint that takes JSON input and returns a classification."

**Good spec:** "A health app user types their weight (kg) and height (cm) and sees their BMI score with a category label like 'normal' or 'overweight'. The result should be accurate to one decimal place."

### The Conversation Process

Work through these questions with the agent before writing any code:

1. **Who is the user?** — A beginner student? An experienced developer? End consumers?
2. **What is the input?** — CLI arguments? File? API request? Interactive form?
3. **What is the output?** — A number? A report? A chart? A downloadable file?
4. **What are the edge cases?** — Invalid input, missing data, extreme values, network failures
5. **What does success look like?** — Concrete acceptance criteria the agent can verify

Example conversation:

```
You: I want to build a tool that helps students check their BMI.

Agent: How will students use it?

You: They'll type their weight in kg and height in cm on the command line.

Agent: What should it show them?

You: Their BMI rounded to one decimal, plus a WHO category. If they enter
invalid data, it should explain the error instead of crashing.

Agent: Should it work as a script or a package they can import?

You: Both — a command-line script and a library they can import.
```

### Lock It In

Once the conversation clarifies what you need, ask the agent to write a **specification document** that captures:

- **Purpose** — one-sentence description of the tool
- **User stories** — "As a [user], I want to [action] so that [benefit]"
- **Input/output** — exact formats, types, and constraints
- **Acceptance criteria** — testable conditions that define "done"
- **Non-goals** — what this version explicitly does NOT do

Save this as `SPEC.md`. It becomes the contract between you and the agent.

## Phase 2: Set Up the Workspace

### Create the Repository

Before the agent writes any code, set up a clean workspace with structure, version control, and safety nets:

```sh
mkdir ~/projects/pybmi && cd ~/projects/pybmi
git init
git config user.name "Your Name"
git config user.email "you@yourdomain.edu"
```

Ask the agent to scaffold:

```
Create a Python package structure:
1. .gitignore for Python
2. pyproject.toml with setuptools and pytest as test dependency
3. pybmi/__init__.py with a docstring
4. tests/test_bmi.py with a placeholder test
5. SPEC.md with our spec
```

Commit everything:

```sh
git add -A && git commit -m "Initial spec and scaffold"
```

**Golden rule:** commit before every agent prompt so you can always revert.

### Isolate the Environment

Create a conda environment so the agent's package installs don't pollute your kernel:

```sh
conda create -n pybmi python=3.12 pip pytest -y
conda activate pybmi
```

Without isolation, the agent might install packages that conflict with your Jupyter kernel. A dedicated environment lets it experiment freely.

### Configure Automated Testing

The agent needs automated feedback to know if it's succeeding. Write tests matching the spec **before** the agent implements anything:

```
In tests/test_bmi.py, add tests matching the spec:
1. test_bmi_calculation — bmi(70, 175) returns approx 22.9
2. test_classify_normal — classify(22.9) returns "normal"
3. test_classify_underweight — classify(15.7) returns "underweight"
4. test_invalid_input — negative weight raises ValueError
```

Run to confirm they **fail** (code doesn't exist yet):

```sh
python -m pytest tests/ -v
# Expected: all FAIL — this is your baseline
```

This is the "Red" phase of TDD. The tests are now the agent's target.

## Phase 3: Agent Implements and Self-Corrects

### The Implementation Prompt

Now give the agent a single, comprehensive prompt that references the spec, tells it to implement one piece at a time, and requires it to run tests after every change:

```
Read SPEC.md and implement the pybmi package.

Follow this process for each function:
1. Write the minimal implementation in pybmi/__init__.py
2. Run pytest to see which tests pass and which fail
3. If tests fail, read the error output and fix the code
4. Repeat until all tests for that function pass
5. Commit before moving to the next function

Stop if you hit an error you cannot resolve after 3 attempts.
```

### How the Agent Works

Here's what happens behind the scenes. The agent follows a loop:

**1. Read the spec** — loads `SPEC.md` to understand requirements

**2. Implement** — writes code based on one spec item

**3. Run tests** — executes `pytest` and reads the output

**4. Analyze failures** — reads traceback, identifies root cause

**5. Fix and repeat** — patches code, re-runs tests

**6. Commit** — saves working code before moving on

The agent sees the same test output you would. If a test fails with `ValueError: height must be positive`, it reads that message and adds input validation. This is the key difference from casual prompting — the agent has **measurable feedback** rather than guessing.

### Watching the Agent in Action

In VSCode (ACP Client), you see the agent's tool calls in real time:

```
[Agent] Reading SPEC.md...
[Agent] Writing pybmi/__init__.py (bmi function)...
[Agent] Running: pytest tests/test_bmi.py -v
[Tool] test_bmi_calculation PASSED
[Tool] test_classify_normal FAILED — NameError: classify not defined
[Agent] classify() not implemented yet, moving on...
[Agent] Writing classify() function...
[Agent] Running: pytest tests/test_bmi.py -v
[Tool] All 3 tests PASSED
[Agent] Committing: git commit -m "feat: implement bmi and classify"
```

In JupyterLab, the chat panel shows the same progression — the agent reports each step as it goes.

### The Agent Hits a Real Bug

Sometimes the agent makes mistakes it can fix itself. Example:

```
[Agent] Running: pytest tests/test_bmi.py -v
[Tool] test_invalid_input FAILED
[Tool] AssertionError: ValueError not raised
[Tool] +  where 0.0 = bmi(-5, 175)
[Agent] The bmi() function doesn't validate negative input.
[Agent] Adding validation:
[Agent]   if weight <= 0: raise ValueError("weight must be positive")
[Agent]   if height <= 0: raise ValueError("height must be positive")
[Agent] Running: pytest tests/test_bmi.py -v
[Tool] All 4 tests PASSED
```

The agent found its own bug, read the test output, and fixed it — without you needing to intervene. This is the **self-healing** loop in action.

## Phase 4: Review and Iterate

### Verify the Results

After the agent finishes, verify independently:

```sh
# Run all tests
python -m pytest tests/ -v

# Try the package manually
python -c "from pybmi import bmi, classify; print(bmi(70, 175), classify(bmi(70, 175)))"

# Review the agent's changes
git log --oneline
git diff HEAD~3
```

### Add Logging for Real-World Debugging

For more complex projects, add logging so the agent (and you) can trace execution when things go wrong:

```python
# pybmi/__init__.py
import logging
logger = logging.getLogger(__name__)

def bmi(weight, height):
    logger.debug(f"Calculating BMI: weight={weight}kg, height={height}cm")
    if weight <= 0:
        logger.error(f"Invalid weight: {weight}")
        raise ValueError("weight must be positive")
    if height <= 0:
        logger.error(f"Invalid height: {height}")
        raise ValueError("height must be positive")
    height_m = height / 100
    result = weight / (height_m ** 2)
    logger.info(f"BMI = {result:.1f}")
    return round(result, 1)
```

When the agent runs tests with `pytest -v -s`, it sees the log output alongside test results. This helps it debug issues that aren't obvious from the traceback alone.

```sh
python -m pytest tests/ -v -s --log-cli-level=DEBUG
```

### Teach the Agent to Read Logs

Include this in your prompt:

```
When tests fail:
1. Read the full pytest output including any print/log statements
2. Identify the root cause (not just the symptom)
3. Check git diff to see what you changed last
4. Fix the root cause, not just the failing line
5. Re-run the full test suite, not just the failing test
```

This trains the agent to debug systematically rather than guessing.

## Putting It All Together

### The Complete Workflow

Here's the full spec-driven workflow in one view:

1. **Chat** — discuss requirements with the agent, focus on the end-user experience
2. **Spec** — lock requirements into `SPEC.md` with acceptance criteria
3. **Scaffold** — create repo, pyproject.toml, environment, and failing tests
4. **Implement** — agent writes code, runs tests, reads errors, fixes bugs
5. **Review** — verify independently, check git history, run manual tests
6. **Iterate** — update spec for new features, repeat from step 4

Each phase feeds the next. The spec guides implementation. The tests verify implementation. The logs help debug implementation. Git tracks all changes.

### Comparison with Casual Prompting

| | Casual Prompting | Spec-Driven Agentic |
|---|---|---|
| **Input** | "Write me a BMI calculator" | SPEC.md with acceptance criteria |
| **Feedback** | You read the code manually | Automated tests + logs |
| **Errors** | You fix them yourself | Agent reads test output and fixes |
| **Tracking** | Copy-paste or manual saves | Git commits after each change |
| **Quality** | Depends on prompt luck | Measured against spec criteria |
| **Scale** | Works for small scripts | Works for multi-file projects |

Spec-driven doesn't mean you can't ask casual questions — it means you have a **safety net** when building anything real.

::::{seealso} Related Guides

- **Hermes Agent** — overview of Hermes, skills, memory, and interfaces
- **writing-plans** skill — how to break specs into bite-sized implementation tasks
- **subagent-driven-development** skill — how to delegate tasks to fresh subagents with review
- **test-driven-development** skill — RED-GREEN-REFACTOR cycle for disciplined coding

::::

::::{exercise} Try It Yourself

Pick a small project and follow the full workflow:

1. Chat with Hermes to define what you want to build (10 min)
2. Ask Hermes to write a SPEC.md (5 min)
3. Set up a repo, conda env, and failing tests (10 min)
4. Give Hermes the implementation prompt and watch it work (15 min)
5. Review the results, check logs, and verify independently (10 min)

Try these project ideas:
- A unit converter (length, temperature, currency)
- A markdown-to-HTML converter
- A simple REST API with FastAPI
- A data visualization tool with matplotlib

Compare the results with casual prompting — you'll notice fewer bugs and more consistent output.

::::